# Intro to substrate saturation curve fitting in Python

 This lesson aims to help fit enzyme substrate saturation curve data and learn a bit of Python coding. If you don't know how to code, don't worry! These lessons assume limited prior knowledge of code or Python.

A few things to start:

1.   These lessons only work in Google Chrome
2.   If you want to save your progress, go to File> Save a Copy in Drive; then locate a spot in your Drive folder

If you have questions, feel free to contact Dr. Chris Berndsen in the JMU Chemistry Department.

---

This colab was written by Dr. Chris Berndsen in the Department of Chemistry and Biochemistry at James Madison University in 2026. Creation of this workflow was in partnership with the [PyBMB community](https://sites.wabash.edu/pybmb/).

---
## Intro to Enzyme Kinetics

Enzyme kinetics is the study of the chemical reactions that are catalyzed by enzymes. It involves understanding how reaction rates are affected by various factors, particularly substrate concentration. The Michaelis-Menten model is a fundamental concept in enzyme kinetics, describing the relationship between the initial reaction rate ($v_0$) and the substrate concentration ([S]) through two key parameters:

$V_{max}$(maximum velocity): The maximum rate of reaction when the enzyme is saturated with substrate.

$K_m$ (Michaelis constant): The substrate concentration at which the reaction rate is half of $V_{max}$. It reflects the affinity of the enzyme for its substrate.

The **Michaelis-Menten equation** is typically non-linear: $V_0 = \frac{V_{max}[S]}{K_m + [S]}$. Fitting experimental data to this equation allows us to determine these crucial kinetic parameters.

###Challenges in Data Fitting

Fitting experimental data to enzyme kinetics models, especially non-linear ones, presents several challenges:



*   **Non-linearity:** Non-linear equations, like
Michaelis-Menten, often require iterative optimization algorithms (like least-squares methods used by curve_fit). These algorithms need good initial estimates to converge effectively and avoid local minima.
*   **Data Quality:** Experimental noise and variability can significantly impact the accuracy of fitted parameters. Outliers can heavily influence the results of least-squares fitting.
*   **Parameter Interdependence:** $V_{max}$ and $K_m$ are often correlated, meaning that an error in estimating one can affect the accuracy of the other.
*  **Transformation Effects:** Historically, linear transformations (like the Lineweaver-Burk plot) were used to simplify fitting. However, these transformations can distort the error structure of the data, giving undue weight to data points with large errors and leading to inaccurate parameter estimates. Modern non-linear fitting methods are generally preferred as they directly fit the raw data without such distortions.
*  **Weighting:** When experimental data points have different levels of uncertainty (e.g., error bars), incorporating this information through weighted fitting can improve the accuracy and precision of the parameter estimates. This is a more advanced technique that accounts for the reliability of each data point.

---


##1. Install Python packages
This notebook utilizes Python, a versatile and widely-used programming language, especially popular in data science and scientific computing. To extend Python's capabilities, we often use external libraries or packages (also known as modules).

In this step, we install essential packages like `pandas` for data manipulation, `numpy` for numerical operations, `scipy.optimize` for curve fitting, `scipy.stats` for statistical functions, matplotlib.pyplot for plotting, and seaborn for advanced data visualization.

When you press the play button next to the code cell, Colab executes the Python code within that cell. This process downloads and installs these packages into the Colab environment, making them available for use in subsequent cells. It might take a minute or two for all installations to complete.

In [ ]:
#@title 1. Install Python packages
#@markdown Press the play button at left and wait a couple of minutes.
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

The code below uses `pandas.read_csv()` to load your data. A .csv (Comma Separated Values) file is a simple text file format used for storing tabular data. Each line in the file represents a row in the table, and values within each row are separated by commas. Replace `xxx.csv` with your file name.

For this analysis, your .csv file should contain two columns:

Substrate Concentration: This column should contain the independent variable (e.g., concentration of a substrate).
Enzyme Rates: This column should contain the dependent variable (e.g., the reaction rate observed at each concentration).
Ensure your file follows this format before proceeding.

In [ ]:
#@title 2. Load the data

# replace xxx with the name of your file
df = pd.read_csv("xxx.csv")
# preview the data, make sure it is three columns of <dbl> type
df.head()


After loading your data, it's good practice to preview it using `df.head()`. This command displays the first few rows of your DataFrame, allowing you to quickly inspect the data's structure, column names, and initial values.



---



The next step ensures your columns are named appropriately for the analysis. For this notebook, we expect the substrate concentration column to be named `conc` and the enzyme rate column to be named `rate`. If your dataset uses different column names, the following code cell will rename them accordingly. If your columns are already named `conc` and `rate`, this step will effectively make no changes.



In [ ]:
#@title 3. Rename the columns using pandas
df = df.rename(columns={'conc':'conc', 'rate': 'rate'})


## Defining the Michaelis-Menten Equation and Initial Estimates

In this step, we define the Michaelis-Menten equation as a Python function `m_m(conc, v, k)`. This function takes the substrate concentration (`conc`) as the independent variable and `v` (Vmax) and `k` (Km) as the parameters to be fitted. The curve_fit function later uses this definition to find the best-fit values for v and k.

We also define `initial_est`, which provides initial guesses for Vmax and Km. These initial estimates are important because `curve_fit` uses them as a starting point for its optimization algorithm. A good guess for Vmax would be the highest rate and for Km it would be the middle concentration value. While not always critical, good initial estimates can help the fitting process converge more quickly and accurately, especially for complex curves.



---

Use the `df.head()` information above to put in guesses for v and k below.

In [ ]:
#@title 4. Define the Michaelis-Menten equation for fitting
def m_m(conc, v, k):
    return (v * conc)/(k + conc)

# define the initial estimates of v and k
# use the df.head information above to put in guesses for v and k
initial_est = [xxx, yyy]

## Performing the Curve Fitting with `curve_fit`
In this step, we use `scipy.optimize.curve_fit` to perform the non-linear least squares fitting. This function attempts to find the optimal parameters for our m_m function that best describe the relationship between the substrate concentration (`df['conc']`) and the enzyme rates (`df['rate']`).

The curve_fit function returns two key objects:

`params`: An array containing the optimal values for the parameters (vmax and km in our case) that result in the best fit. These are the values that minimize the sum of the squared residuals.

`cov`: The estimated covariance matrix of the optimized parameters. The diagonal elements of this matrix represent the variance of each parameter, from which we can calculate the standard errors (square root of the diagonal elements). These errors provide an indication of the precision of our fitted parameters.


In [ ]:
#@title 5. Fit the data
params, cov = curve_fit(m_m, # equation
                        df['conc'], # independent variable
                        df['rate'], # dependent variable
                        p0 = initial_est # initial estimates
                       )

## Extracting Fitted Parameters and Their Errors
After successfully fitting the curve, the next step is to extract the individual parameter values and their corresponding errors from the params and cov objects.

`vmax, km = params`: This line unpacks the params array into two separate variables: vmax and km. The curve_fit function returns parameters in the order they appear in the defined function (m_m in this case), so vmax is the first parameter and km is the second.

`errors = np.sqrt(np.diag(cov))`: This calculates the standard errors for each fitted parameter. The cov matrix contains the variance of each parameter on its diagonal. Taking the square root of these diagonal elements (`np.diag(cov)`) gives the standard deviation, which represents the standard error of the estimate.

print statements and `.4f`: The print statements are used to display the calculated Vmax, Km, and their respective errors in a human-readable format. The .4f format specifier inside the f-string (`f"Vmax = {vmax:.4f}"`) is used to format the floating-point numbers to four decimal places, ensuring consistent and clear output.

*Aside: what is an f-string?: An f-string (formatted string literal) is a way to embed expressions in Python. It's a concise and readable way to create formatted strings. This this case we have it print the Vmax without having to type the number and indicate the number of decimal places in the number. [For further reading](https://cissandbox.bentley.edu/sandbox/wp-content/uploads/2022-02-10-Documentation-on-f-strings-Updated.pdf).*

In [ ]:
#@title 6a. Extract the values and errors from params and cov
vmax, km = params

errors = np.sqrt(np.diag(cov))

# Print out the values and errors in human readable format
print("Fitted Michaelis-Menten values:")

print(f"Vmax = {vmax:.4f}", f"error = {errors[0]:.4f}")
print(f"Km = {km:.4f}", f"error = {errors[1]:.4f}")


## Generating the Fitted Line for Plotting
After obtaining the optimal Vmax and Km values from `curve_fit`, we need to generate a smooth curve that represents the fitted Michaelis-Menten equation across a range of substrate concentrations. This fitted line is then plotted along with the experimental data points.

`c_range = np.linspace(df['conc'].min()-1, df['conc'].max()+1, 200)`: This line uses `numpy.linspace` to create an array of evenly spaced numbers, representing a continuous range of substrate concentrations.
The range (`df['conc'].min()-1` to `df['conc'].max()+1`) is chosen to extend slightly beyond the minimum and maximum experimental concentrations. This helps ensure the fitted curve is drawn smoothly across and slightly beyond the observed data points, providing a complete visual representation.

`rate_fit = m_m(c_range, vmax, km)`:

This line utilizes the `m_m` function (our defined Michaelis-Menten equation from block 4).
It takes the generated `c_range` (the array of substrate concentrations) as the input independent variable.
Crucially, it uses the vmax and km values that were determined by the `curve_fit` function. These are the best-fit parameters derived from your experimental data.
The output, `rate_fit`, is an array of predicted reaction rates corresponding to each concentration in `c_range`, forming the smooth Michaelis-Menten curve that best fits the data.



---


Below, change the xxx to at least 200, to generate 200 concentrations. Come back and play with more or fewer points to see how this changes the fit and plot.

In [ ]:
#@title 6b. Create the fitted line using Vmax and Km from `curve_fit`
# use the fit to generate data to create the fitting line in the plot
c_range = np.linspace(df['conc'].min()-1, # lower limit
                      df['conc'].max()+1, # upper limit
                      xxx) # number of points between the limits

rate_fit = m_m(c_range, # s values to use in the prediction
               vmax, # from the fit, v in the function
               km) # from the fit, k in the function

##Visualizing the Michaelis-Menten Fit
This section focuses on visualizing the experimental data and the fitted Michaelis-Menten curve. We use two powerful Python libraries for plotting:

`matplotlib.pyplot` (imported as `plt`): This is the foundational plotting library in Python, providing a wide array of functions for creating static, interactive, and animated visualizations. Here, it's used for overall figure management (`plt.figure`, `plt.xlabel`, `plt.ylabel`, `plt.title`, `plt.legend`, `plt.grid`, `plt.tight_layout`, `plt.show`).

`seaborn` (imported as `sns`): Built on top of matplotlib, seaborn provides a high-level interface for drawing attractive and informative statistical graphics. We use `sns.scatterplot()` to display the individual data points (substrate concentration vs. reaction rate), making it easy to see the raw experimental results. The `s`, `color`, `edgecolor`, `linewidth`, and `zorder` arguments are used to customize the appearance of these points.

The fitted curve is then plotted using `plt.plot()`, drawing a smooth line that represents the Michaelis-Menten equation with the optimized Vmax and Km values. This visual comparison helps assess how well the model describes the experimental data.

In [ ]:
#@title 7. Create the plot
plt.figure(figsize=(10, 6))

# Plot the original data points
sns.scatterplot(x=df['conc'], y=df['rate'], s=100, color='#2E86AB',
                label='Observed Data', edgecolor='black', linewidth=1.5, zorder=3)

# Plot the fitted curve
plt.plot(c_range, rate_fit, color='#A23B72', linewidth=2.5,
         label=f'Fitted Curve\nVmax = {vmax:.2f}\nKm = {km:.2f}', zorder=2)

# Add labels and title
plt.xlabel('Substrate Concentration', fontsize=12, fontweight='bold')
plt.ylabel('Reaction Rate', fontsize=12, fontweight='bold')
plt.title('Michaelis-Menten Enzyme Kinetics', fontsize=14, fontweight='bold', pad=20)

# Add legend
plt.legend(fontsize=10, loc='lower right', frameon=True, shadow=True)

# Add grid
plt.grid(True, alpha=0.3, linestyle='--')

# Tight layout
plt.tight_layout()

# Show plot
plt.show()

## Mathematical Transformations for Plotting
Sometimes, to analyze data in different ways or to linearize non-linear relationships, we apply mathematical transformations to our data. A common example in enzyme kinetics is the Lineweaver-Burk plot, which involves taking the reciprocal of both substrate concentration and reaction rate.

In Python, using libraries like `matplotlib` and `seaborn`, we can perform these mathematical transformations directly when plotting. Instead of creating new DataFrame columns for transformed data, we can apply operations (like `1/df['conc']`) directly within the plotting function calls for the x and y axes. This allows for flexible visualization of transformed data without altering the original dataset.



---
Change the x and y values in the `scatterplot` and `plot` to be the reciprocal values (e.g. 1/conc). One caution, `'conc'` is the title of the column. The actual value is `df['conc']` so be aware of this when adding the 1/ in the code.


In [ ]:
#@title 8. Convert data to Lineweaver-Burk format
# Create the plot
plt.figure(figsize=(10, 6))

# Plot the original data points
sns.scatterplot(x=df['conc'], y=df['rate'], s=100, color='#2E86AB',
                label='Observed Data', edgecolor='black', linewidth=1.5, zorder=3)

# Plot the fitted curve
plt.plot(c_range, rate_fit, color='#A23B72', linewidth=2.5,
         label=f'Fitted Curve\nVmax = {vmax:.2f}\nKm = {km:.2f}', zorder=2)

# Add labels and title
plt.xlabel('Substrate Concentration', fontsize=12, fontweight='bold')
plt.ylabel('Reaction Rate', fontsize=12, fontweight='bold')
plt.title('Michaelis-Menten Enzyme Kinetics', fontsize=14, fontweight='bold', pad=20)

# Add legend
plt.legend(fontsize=10, loc='lower right', frameon=True, shadow=True)

# Add grid
plt.grid(True, alpha=0.3, linestyle='--')

# Tight layout
plt.tight_layout()

# Show plot
plt.show()

While it is easy to convert the data to Lineweaver-Burk format in the plot, we did not fit the transformed data. Since drawing straight lines is easier, why would we not fit the data in this format instead? Let's explore that topic below.


---



## Understanding Lineweaver-Burk Linear Regression
In the next step, we will perform a linear regression on the Lineweaver-Burk transformed data using `scipy.stats.linregress`. This function is specifically designed for simple linear regression and returns several statistical values:

`slope`: The slope of the regression line.

`intercept`: The y-intercept of the regression line.

`r_value`: The Pearson correlation coefficient.

`p_value`: The two-sided p-value for a hypothesis test whose null hypothesis is that the slope is zero.

`std_err`: The standard error of the estimated slope.



---



For the Lineweaver-Burk plot, the equation is: 1/rate = (Km/Vmax) * (1/conc) + 1/Vmax.

By comparing this to the standard linear equation y = mx + b:

The slope (m) corresponds to Km/Vmax.
The intercept (b) corresponds to 1/Vmax.
From these relationships, we can calculate Vmax and Km:

`vmax_lb = 1 / intercept`: Since the y-intercept is 1/Vmax, Vmax is simply the reciprocal of the intercept.

`km_lb = slope * vmax_lb`: Since the slope is Km/Vmax, we can rearrange this to find Km: Km = slope * Vmax. We use the `vmax_lb` value we just calculated.

In [ ]:
#@title 9a. Fit the data in L-B format using linear regression

# remember we use inverse values
slope, intercept, r_value, p_value, std_err = stats.linregress(1/df['conc'], 1/df['rate'])

# Calculate Vmax and Km from the linear fit
vmax_lb = 1 / intercept
km_lb = slope * vmax_lb

# calculate Vmax and Km from L-B fitting
print(vmax_lb)
print(km_lb)

## An alternative fitting method
Below is code to fit Lineweaver-Burk form data using `curve_fit` rather than `linregress`. *In theory*, this will give the same results as using `linregress`.



---
In the code below, replace the `xxxx` and `yyyy` with the slope and intercept variables from `curve_fit`.


In [ ]:
#@title 9b. L-B fitting with `curve_fit`
# Define a linear function for Lineweaver-Burk plot
def lineweaver_burk_linear(inv_conc, slope, intercept):
    return slope * inv_conc + intercept

# Perform curve_fit on the reciprocal data
params_lb, cov_lb = curve_fit(lineweaver_burk_linear,
                               1/df['conc'],
                               1/df['rate'])

# Extract slope and intercept from curve_fit results
slope_cf, intercept_cf = params_lb

# Calculate Vmax and Km from the linear fit parameters
vmax_lb_cf = 1 / xxxx
km_lb_cf = yyyy * xxxx

print("Fitted Lineweaver-Burk values (using curve_fit):")
print(f"Vmax = {vmax_lb_cf:.4f}")
print(f"Km = {km_lb_cf:.4f}")

As we did above, we need to generate a fitted data set using the slope and intercept values from the Lineweaver-Burk fit. We will use the values from `linregress` in 9a, however you can alter this OR create an additional data set using the `curve_fit` values. This could be useful if you saw differences between the values in the two methods. To use the oter methods, alter the `slope` and `intercept` in the `inv_rate_fitted` value in the last line.

In [ ]:
#@title 10. Generate line for plotting

# remember we need to use the inverse values!
inv_conc_range = np.linspace(1/df['conc'].min(),
                             1/df['conc'].max(),
                             200)

inv_rate_fitted = slope * inv_conc_range + intercept

Now plot the data fitted directly to the data and the fit of the Lineweaver-Burk data form to see how comparable the data fits are. Ideally, they will be identical. Notice, we only added a `plt.plot` to show this new data set. Thus, it is fairly easy to compare different fits and data sets using the same plot code.

In [ ]:
#@title 11. Plotting the Lineweaver-Burk format data
# Create the plot
plt.figure(figsize=(10, 6))

# Plot the original data points in LB format
sns.scatterplot(x=1/df['conc'], y=1/df['rate'], s=100, color='#2E86AB',
                label='Observed Data', edgecolor='black', linewidth=1.5, zorder=3)

# Plot the fitted curve in LB format
plt.plot(1/c_range, 1/rate_fit, color='#A23B72', linewidth=2.5,
         label=f'Fitted Curve from MM\nVmax = {vmax:.2f}\nKm = {km:.2f}', zorder=2)

# Plot the fitted curve from the linear fit of the reciprocal data
plt.plot(inv_conc_range, inv_rate_fitted, linewidth=2.5,
         label=f'Fitted Curve from LB\nVmax = {vmax_lb:.2f}\nKm = {km_lb:.2f}', zorder=2)

# Add labels and title
plt.xlabel('1/Substrate Concentration', fontsize=12, fontweight='bold')
plt.ylabel('1/Reaction Rate', fontsize=12, fontweight='bold')
plt.title('Comparing Michaelis-Menten and LB fitting', fontsize=14, fontweight='bold', pad=20)

# Add legend
plt.legend(fontsize=10, loc='lower right', frameon=True, shadow=True)

# Add grid
plt.grid(True, alpha=0.3, linestyle='--')

# Tight layout
plt.tight_layout()

# Show plot
plt.show()

It is likely that the L-B fit to the data will not match the direct fit. This is an inherent flaw of non-linear transformations to data, which often skew data. Broadly speaking, Lineweaver-Burk fitting is good for qualitatively comparisons of data, but not great for getting the numbers. With the advent of computers and software like Python or even Excel, we can fit the data without needing to do data transformations.

Going forward, you can use the code in blocks 1-7 to fit data and you do not need to use Lineweaver-Burk fitting.



---

The Challenges below will help you explore Python coding and alternative fitting methods using the packages.


In [ ]:
#@title CHALLENGE #1: Convert LB data to MM data

# Make a data plot of rate (y) vs. concentration (x) converting the LB fit to the MM style plot to compare the methods
## HINT: This can be accomplished without ANY addtional fitting, simply modifying the plot code above. Remember how we converted the MM data to LB format?


In [ ]:
#@title CHALLENGE #2: Add a residuals plot to the Michaelis-Menten plot

# The calculation of the residuals is already done
# You need to modify the code to 1: add a second plot to the figure call (ax2) and 2: create a scatter plot of residuals vs. concentration
# HINT: you will need to add a axhline with y = 0 called the zero line in addition to the labels and title. Remember the residuals is ax2 throughout

# Create a figure with two subplots
fig, (ax1, ___) = plt.subplots(2, 1, figsize=(10, 10),
                                gridspec_kw={'height_ratios': [3, 1]})

# Top plot: Data and fit
sns.scatterplot(x=df['conc'], y=df['rate'], s=100, color='#2E86AB',
                label='Observed Data', edgecolor='black', linewidth=1.5,
                ax=ax1, zorder=3)
ax1.plot(c_range, rate_fit, color='#A23B72', linewidth=2.5,
         label=f'Fitted Curve (Vmax={vmax:.2f}, Km={km:.2f})', zorder=2)
ax1.set_xlabel('Substrate Concentration', fontsize=12, fontweight='bold')
ax1.set_ylabel('Reaction Rate', fontsize=12, fontweight='bold')
ax1.set_title('Michaelis-Menten Enzyme Kinetics', fontsize=14, fontweight='bold', pad=20)
ax1.legend(fontsize=10, loc='lower right', frameon=True, shadow=True)
ax1.grid(True, alpha=0.3, linestyle='--')

## Calculate the residuals
predicted_rates = m_m(df['conc'], vmax, km)
residuals = df['rate'] - predicted_rates

# Bottom plot: Residuals plot
___.scatter(________, ______, s=100, color='#F18F01',
            edgecolor='black', linewidth=1.5, zorder=3)
# add in the horizontal line using axhline with y = 0 inside the function

# add in the labels, title, legend, and grid calls


plt.tight_layout()
plt.show()

In [ ]:
#@title CHALLENGE #3: Incorporate error into the fitting

# Read the documentation for `curve_fit` online and fit XXXX.csv
# incorporating the experimental uncertainty found in the error column of the data
## an outline of the steps is defined below

# read the .csv and call it df


# make sure the column names are machine readable


# define the MM equation and initial estimates



# fit the data to the curve using the initial estimates as a starting point and weighting the data by the error column



# extract the values and errors from params and cov



# print out the values



# make the plot

In [ ]:
#@title CHALLENGE #4: Fit data using the Hanes-Woolf method

# Fit the data using the Hanes-Woolf method and make a plot
# Hanes-Woolf form of the MM equation: conc/rate = conc/Vmax + Km/Vmax
# Hanes-Woolf is a linear transformation of the data similar to Lineweaver-Burk format
# Slope is 1/Vmax, y-intercept is Km/Vmax, and x-intercept is -Km
## HINT: There are TWO ways to accomplish the fitting